# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Musadiq8699/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The Scoring Rule (Plain Words)
An article's priority score (0.0 to 1.0) is determined by the absolute search volume at risk multiplied by its rate of decay. Only eligible articles with a minimum baseline of 10 prior clicks or 500 prior impressions are evaluated; dead or dormant pages receive a score of 0.0.

For eligible pages, the baseline score scales with the absolute clicks lost, weighted by how severe the decay is:

$$\text{Action Score} = \min\left(1.0, \, \frac{\max(0, \, \text{clicks\_prev\_30d} - \text{clicks\_last\_30d})}{50} \times (1.0 - \text{click\_decay\_ratio})\right)$$

### Reason Codes (Diagnoses & Treatments)
* **`CRITICAL_TRAFFIC_DECAY`**: Triggered when `click_decay_ratio < 0.50` on pages with $\ge 10$ prior clicks. Identifies fading articles that require a freshness content refresh.
* **`SNIPPET_FIX_CTR_GAP`**: Triggered when a page has high search visibility (`impressions_last_30d >= 500` and `position <= 15`) but near-zero clicks (`CTR < 0.5%`). The shop window failed; title tag and meta description rewrite needed.
* **`PAGE_ONE_PUSH`**: Triggered when an article sits on Page 2 (`position` between 11.0 and 20.0) with steady impressions. Content expansion and interlinking needed to push onto Page 1.
* **`NO_ACTION_REQUIRED`**: Assigned to healthy, growing, or dormant low-traffic pages that do not warrant human sprint capacity.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

def compute_baseline_score_and_reasons(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes a deterministic baseline score (0.0 to 1.0) and assigns
    diagnostic reason codes based on Haris's decision framework.
    """
    df = df.copy()

    # 1. Traffic delta & decay rate
    clicks_lost = np.maximum(0, df['clicks_prev_30d'] - df['clicks_last_30d'])
    decay_intensity = np.clip(1.0 - df['click_decay_ratio'], 0.0, 1.0)

    # 2. Eligible population filter (at least 10 prior clicks or 500 impressions)
    eligible = (df['clicks_prev_30d'] >= 10) | (df['impressions_last_30d'] >= 500)

    # 3. Action Score formula (scaled to max out at 50 lost clicks with full decay)
    raw_score = (clicks_lost / 50.0) * decay_intensity
    df['action_score'] = np.where(eligible, np.clip(raw_score, 0.0, 1.0), 0.0)

    # Calculate CTR
    ctr = df['clicks_last_30d'] / (df['impressions_last_30d'] + 1e-5)

    # 4. Diagnostic Reason Codes
    conditions = [
        (~eligible),
        (df['click_decay_ratio'] < 0.50) & (df['clicks_prev_30d'] >= 10),
        (df['impressions_last_30d'] >= 500) & (df['position_last_30d'] <= 15.0) & (ctr < 0.005),
        (df['position_last_30d'] > 10.0) & (df['position_last_30d'] <= 20.0) & (df['impressions_last_30d'] >= 200)
    ]

    choices = [
        'NO_ACTION_REQUIRED',
        'CRITICAL_TRAFFIC_DECAY',
        'SNIPPET_FIX_CTR_GAP',
        'PAGE_ONE_PUSH'
    ]

    df['reason_code'] = np.select(conditions, choices, default='NO_ACTION_REQUIRED')
    return df

# Unit test verification
test_df = pd.DataFrame([
    {'content_hash_id': 'article_c_fading', 'clicks_prev_30d': 120, 'clicks_last_30d': 30, 'impressions_prev_30d': 3000, 'impressions_last_30d': 1200, 'click_decay_ratio': 0.25, 'position_last_30d': 8.0},
    {'content_hash_id': 'article_a_snippet', 'clicks_prev_30d': 60, 'clicks_last_30d': 5, 'impressions_prev_30d': 12000, 'impressions_last_30d': 12000, 'click_decay_ratio': 0.08, 'position_last_30d': 7.0},
    {'content_hash_id': 'article_d_page2', 'clicks_prev_30d': 15, 'clicks_last_30d': 16, 'impressions_prev_30d': 800, 'impressions_last_30d': 850, 'click_decay_ratio': 1.06, 'position_last_30d': 12.3},
    {'content_hash_id': 'article_dormant', 'clicks_prev_30d': 1, 'clicks_last_30d': 0, 'impressions_prev_30d': 5, 'impressions_last_30d': 2, 'click_decay_ratio': 0.0, 'position_last_30d': 65.0}
])

print(compute_baseline_score_and_reasons(test_df)[['content_hash_id', 'action_score', 'reason_code']])

     content_hash_id  action_score             reason_code
0   article_c_fading           1.0  CRITICAL_TRAFFIC_DECAY
1  article_a_snippet           1.0  CRITICAL_TRAFFIC_DECAY
2    article_d_page2           0.0           PAGE_ONE_PUSH
3    article_dormant           0.0      NO_ACTION_REQUIRED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

# 1. Connect & aggregate warehouse data
HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

api = HfApi()
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
fact_files = [f for f in repo_files if f.startswith("fact_content_daily_performance/") and f.endswith(".parquet")]

local_paths = [
    hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN)
    for f in fact_files
]

con = duckdb.connect()
con.execute(f"CREATE VIEW fact_table AS SELECT * FROM read_parquet({local_paths})")

# SQL: 30-day snapshot vs previous 30-day baseline per content_hash_id
query = """
WITH max_date_cte AS (SELECT MAX(report_date) AS max_date FROM fact_table),
page_agg AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days' THEN gsc_clicks ELSE 0 END) AS clicks_last_30d,
        SUM(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days' THEN gsc_impressions ELSE 0 END) AS impressions_last_30d,
        AVG(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days' THEN gsc_avg_position ELSE NULL END) AS position_last_30d,
        SUM(CASE WHEN report_date < (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                  AND report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '60 days' THEN gsc_clicks ELSE 0 END) AS clicks_prev_30d,
        SUM(CASE WHEN report_date < (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                  AND report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '60 days' THEN gsc_impressions ELSE 0 END) AS impressions_prev_30d
    FROM fact_table
    GROUP BY content_hash_id
)
SELECT * FROM page_agg;
"""

df_full = con.execute(query).df()

# 2. Compute decay ratios
df_full['click_decay_ratio'] = (df_full['clicks_last_30d'] / (df_full['clicks_prev_30d'] + 1e-5)).replace([np.inf, -np.inf], np.nan).fillna(1.0)
df_full['position_last_30d'] = df_full['position_last_30d'].fillna(df_full['position_last_30d'].median())

# 3. Apply scoring function & sort descending
df_scored = compute_baseline_score_and_reasons(df_full)
df_ranked = df_scored.sort_values(by='action_score', ascending=False).reset_index(drop=True)

# 4. Save output CSV
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "baseline_action_score.csv")

# Export relevant columns for submission
df_ranked[['content_hash_id', 'action_score', 'reason_code', 'clicks_last_30d', 'clicks_prev_30d', 'impressions_last_30d', 'click_decay_ratio']].to_csv(output_path, index=False)

print("=== BASELINE ACTION QUEUE GENERATED ===")
print(f"Total pages scored: {len(df_ranked):,}")
print(f"Actionable pages (action_score > 0): {(df_ranked['action_score'] > 0).sum():,}")
print(f"CSV successfully exported to: {output_path}")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== BASELINE ACTION QUEUE GENERATED ===
Total pages scored: 427,292
Actionable pages (action_score > 0): 24,462
CSV successfully exported to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.